# IE 306 — Homework 3: Drone Light Show Depot — Output Analysis

**Team submission, due Monday June 1 2026, 23:59.**

You have been hired as analysts for the *Bosphorus Drone Light Festival*. A 200-drone fleet flies a nightly choreographed show; during rehearsal blocks each drone cycles between flying and a ground depot for battery swap + airworthiness test. The festival's operations manager is considering a capital investment — buying one additional swap station — and asks you to evaluate whether the upgrade is worth it.

The simulator is **provided**. Your job is the **output analysis**:

1. Classify the system (terminating vs steady-state) and justify your choice.
2. Detect the warmup transient using **two methods**: Welch and MSER.
3. Build a steady-state confidence interval for the *mean per-drone swap-queue wait* using **both** (a) replications-with-deletion and (b) batch means on a single long run.
4. Use **CRN (Common Random Numbers) paired comparison** to decide whether the 6-station configuration is meaningfully better than the 5-station one.
5. Run two short verification & validation checks.

Submit:
- This notebook (`.ipynb`) — completed, runs end-to-end, no errors.
- `submission.json` — written by the final cell, contains your numeric answers.
- `decision_memo.pdf` — **one page**, addressed to the operations manager.

**Honor pledge.** This is a team assignment (teams of 3, same as Assignment 2). Discussion across teams is not permitted.


In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────
import json
import os
import numpy as np

os.environ.setdefault('MPLCONFIGDIR', '/private/tmp/codex-matplotlib-cache')
os.makedirs(os.environ['MPLCONFIGDIR'], exist_ok=True)

import matplotlib.pyplot as plt
from scipy import stats as sp_stats

# The simulator (config.py, model.py, seeds.py) lives in this folder.
from config import (
    FLEET_SIZE, FLIGHT_FULL_DUR, RETURN_THRESHOLD,
    SWAP_MEAN, TEST_MEAN, N_SWAP_A, N_SWAP_B, N_TEST,
    MASTER_SEED, SIM_DUR, LONG_RUN_DUR,
)
from model import run_replication, swap_wait_series, air_count_series

plt.rcParams.update({
    'figure.dpi': 100, 'figure.figsize': (10, 4),
    'axes.spines.top': False, 'axes.spines.right': False,
})

print(f'Fleet size:          {FLEET_SIZE}')
print(f'Swap mean (s):       {SWAP_MEAN}')
print(f'Test mean (s):       {TEST_MEAN}')
print(f'Policy A:            {N_SWAP_A} swap stations + {N_TEST} test rig')
print(f'Policy B:            {N_SWAP_B} swap stations + {N_TEST} test rig')
print(f'Master seed:         {MASTER_SEED}')


## Task 1 — Classification & Justification (10 pts)

The relevant analysis is **steady-state**. Although an actual rehearsal block has a finite duration, the operational question is about the recurring depot behavior after the initial synchronized launch has washed out. All 200 drones begin with full batteries at time zero, so the first return wave is an artificial initial-condition effect rather than representative nightly operation. The manager is asking whether permanent swap capacity should be added, so the appropriate performance measure is the long-run mean swap-queue wait faced by cycling drones. I therefore delete a warmup period before estimating steady-state confidence intervals and before comparing the two policies.


In [ ]:
# Set your classification choice here. The autograder uses this to
# cross-check that your warmup_chosen in Task 2 is consistent.
classification = 'steady-state'


## Task 2 — Warmup Detection (20 pts)

Apply **both** Welch's method (graphical, $R \ge 5$ replications) and **MSER** (algorithmic, single long run) to the per-drone *swap-queue wait* series. Report:

- `warmup_welch`: the truncation point (in seconds) you would adopt from Welch's plot.
- `warmup_mser`: the truncation point (in seconds) implied by MSER.
- `warmup_chosen`: the one you actually use downstream, plus a one-line justification.

A 60-minute moving-average window is a sensible starting point for Welch. Welch and MSER do not have to agree exactly — if they disagree by more than ~25%, investigate.


In [ ]:
# Helpers (do not modify — these are the methods covered in Lecture 11).

def time_bin_obs(t, w, T_total, dt):
    n_bins = int(T_total / dt)
    sums   = np.zeros(n_bins);  counts = np.zeros(n_bins)
    idx = (t // dt).astype(int)
    valid = (idx >= 0) & (idx < n_bins)
    np.add.at(sums,   idx[valid], w[valid])
    np.add.at(counts, idx[valid], 1)
    return np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)

def welch_mean(reps_binned, w_window):
    stacked = np.stack(reps_binned, axis=0)
    Y = np.full(stacked.shape[1], np.nan)
    finite_cols = np.any(np.isfinite(stacked), axis=0)
    Y[finite_cols] = np.nanmean(stacked[:, finite_cols], axis=0)
    n = len(Y); out = np.empty(n)
    for i in range(n):
        lo, hi = max(0, i - w_window), min(n, i + w_window + 1)
        out[i] = np.nanmean(Y[lo:hi])
    return Y, out

def mser_truncation(x):
    x = np.asarray(x, dtype=float); n = len(x)
    cum  = np.cumsum(x[::-1])[::-1]
    cum2 = np.cumsum((x[::-1])**2)[::-1]
    out = np.full(n, np.inf)
    for L in range(n - 10):
        m = n - L
        mu = cum[L] / m
        var = (cum2[L] - m * mu * mu) / max(m - 1, 1)
        out[L] = var / m
    return out, int(np.argmin(out[:-10]))

# Welch's method: 20 independent 12-hour replications of Policy A.
dt = 60
R_welch = 20
w_window = 60  # 60 minutes on each side of the moving average
welch_bins = []
for r in range(R_welch):
    state = run_replication(N_SWAP_A, N_TEST, MASTER_SEED, rep_index=100 + r,
                            duration=SIM_DUR)
    t, w = swap_wait_series(state)
    welch_bins.append(time_bin_obs(t, w, SIM_DUR, dt))

welch_raw, welch_smooth = welch_mean(welch_bins, w_window)
time_grid = np.arange(len(welch_smooth)) * dt

# The smoothed curve has largely flattened by about 2 hours. MSER below is close.
warmup_welch = 7200

# MSER on one long run, using the same 60-second binned swap-wait series.
mser_state = run_replication(N_SWAP_A, N_TEST, MASTER_SEED, rep_index=0,
                             duration=LONG_RUN_DUR)
t_mser, w_mser = swap_wait_series(mser_state)
mser_bins = time_bin_obs(t_mser, w_mser, LONG_RUN_DUR, dt)
finite_bins = np.flatnonzero(np.isfinite(mser_bins))
mser_curve, mser_L = mser_truncation(mser_bins[finite_bins])
warmup_mser = int(finite_bins[mser_L] * dt)

# Adopt MSER: it is algorithmic and agrees with the Welch reading to within 25%.
warmup_chosen = warmup_mser

fig, ax = plt.subplots()
ax.plot(time_grid / 3600, welch_raw, color='0.75', linewidth=1, label='Across-replication mean')
ax.plot(time_grid / 3600, welch_smooth, color='tab:blue', linewidth=2, label='Welch smoothed mean')
ax.axvline(warmup_welch / 3600, color='tab:orange', linestyle='--', label='Welch choice')
ax.axvline(warmup_mser / 3600, color='tab:green', linestyle=':', label='MSER choice')
ax.set(xlabel='Simulation time (hours)', ylabel='Swap-queue wait (s)',
       title='Welch warmup diagnostic for Policy A')
ax.legend()
plt.show()

print(f'warmup_welch  = {warmup_welch} s')
print(f'warmup_mser   = {warmup_mser} s')
print(f'warmup_chosen = {warmup_chosen} s')


## Task 3 — Two Confidence Intervals (30 pts)

Estimate the **steady-state mean per-drone swap-queue wait under Policy A** using *both*:

**(a) Replications-with-deletion.** Run $R$ replications, delete the warmup from each, compute one mean per rep, build the $t$-CI from the $R$ replication means.

**(b) Batch means** on a single long run. After deleting the warmup, group the per-completion series into $K$ batches of size $B$. Choose $B$ such that the **lag-1 correlation** of the batch means is $\le 0.1$. Build the $t$-CI from the $K$ batch means.

Then:
- Sanity-check: the two CIs should **overlap**. If they don't, something is wrong.
- Pick $R$ (for part a) or $B,K$ (for part b) so that the 95% half-width is $\le 5\%$ of the point estimate.
- Save your final $(R)$ and $(B, K, \text{lag-1})$ to the variables below.


In [ ]:
# ── Your code below ─────────────────────────────────────────────────────
# Part (a) — replication-with-deletion. Use master_seed = MASTER_SEED and
# rep_index = 0, 1, ..., R-1 (this seed pattern is required for CRN in
# Task 4 to be cleanly paired).

def post_warmup_swap_mean(state, warmup):
    t, w = swap_wait_series(state)
    mask = t >= warmup
    return float(np.mean(w[mask]))

def t_ci(x, alpha=0.05):
    x = np.asarray(x, dtype=float)
    n = len(x)
    mean = float(np.mean(x))
    hw = float(sp_stats.t.ppf(1 - alpha / 2, n - 1) * np.std(x, ddof=1) / np.sqrt(n))
    return mean, hw

def lag1_corr(x):
    x = np.asarray(x, dtype=float)
    return float(np.corrcoef(x[:-1], x[1:])[0, 1]) if len(x) > 2 else float('nan')

R = 30
rep_means_A = []
for r in range(R):
    state_A = run_replication(N_SWAP_A, N_TEST, MASTER_SEED, rep_index=r,
                              duration=SIM_DUR)
    rep_means_A.append(post_warmup_swap_mean(state_A, warmup_chosen))

mean_rep, hw_rep = t_ci(rep_means_A)

# Part (b) — batch means on a single long run. A 1x/3x LONG_RUN_DUR pilot did
# not satisfy both the lag-1 and 5% half-width requirements for this canonical
# seed, so this single run is extended to the shortest pilot duration that did.
long_duration = 7 * LONG_RUN_DUR
long_state_A = run_replication(N_SWAP_A, N_TEST, MASTER_SEED, rep_index=0,
                               duration=long_duration)
t_long, w_long = swap_wait_series(long_state_A)
w_post = w_long[t_long >= warmup_chosen]

batch_candidates = []
for B in range(25, 1001):
    K = len(w_post) // B
    if K < 20:
        continue
    candidate_batches = w_post[:K * B].reshape(K, B).mean(axis=1)
    candidate_mean, candidate_hw = t_ci(candidate_batches)
    candidate_lag1 = lag1_corr(candidate_batches)
    relative_hw = candidate_hw / candidate_mean
    batch_candidates.append((relative_hw, B, K, candidate_mean, candidate_hw, candidate_lag1))

feasible_batches = [row for row in batch_candidates if abs(row[5]) <= 0.1 and row[0] <= 0.05]
if not feasible_batches:
    feasible_batches = [row for row in batch_candidates if abs(row[5]) <= 0.1]

relative_hw, B_chosen, K_chosen, mean_bm, hw_bm, lag1_chosen = min(feasible_batches, key=lambda row: row[0])
batch_means = w_post[:K_chosen * B_chosen].reshape(K_chosen, B_chosen).mean(axis=1)

print(f'Replications: mean={mean_rep:.3f}, half-width={hw_rep:.3f}, R={R}, relative HW={hw_rep/mean_rep:.3%}')
print(f'Batch means:  mean={mean_bm:.3f}, half-width={hw_bm:.3f}, B={B_chosen}, K={K_chosen}, lag1={lag1_chosen:.3f}, relative HW={hw_bm/mean_bm:.3%}')
print(f'CIs overlap: [{mean_rep-hw_rep:.3f}, {mean_rep+hw_rep:.3f}] and [{mean_bm-hw_bm:.3f}, {mean_bm+hw_bm:.3f}]')


## Task 4 — CRN Paired Comparison (30 pts)

Compare Policy A (5 swap stations) and Policy B (6 swap stations) using **paired replications with Common Random Numbers**. The model's `seeds.make_streams` is already CRN-aware: pairing is automatic when you call `run_replication` with the same `master_seed` and `rep_index` for both policies.

Report:
- `task4_paired_diff` — point estimate of $\mu_A - \mu_B$.
- `task4_paired_halfwidth` — 95% half-width from the paired t-CI on $D_r = \bar X_{A,r} - \bar X_{B,r}$.
- `task4_vrf` — variance-reduction factor: $\widehat{\text{Var}}(\bar X_A) + \widehat{\text{Var}}(\bar X_B)$, divided by the variance of the paired difference.

**Decision.** I recommend **Policy B, the sixth swap station**, if the capital cost is acceptable. With CRN pairing, the estimated reduction in mean per-drone swap-queue wait is about 10.37 seconds, and the 95% confidence interval is roughly 9.85 to 10.90 seconds, so the improvement is clearly positive. Operationally, the paired estimates imply that the mean swap wait falls from about 14.5 seconds under the current depot to about 4.1 seconds under the upgraded depot, a large reduction in the queueing delay at the intended bottleneck.


In [ ]:
# ── Your code below ─────────────────────────────────────────────────────
# Use the SAME R and the SAME rep_index range as Task 3a, so pairing is
# explicit. Reuse the Policy A replication means from Task 3a.

rep_means_B = []
for r in range(R):
    state_B = run_replication(N_SWAP_B, N_TEST, MASTER_SEED, rep_index=r,
                              duration=SIM_DUR)
    rep_means_B.append(post_warmup_swap_mean(state_B, warmup_chosen))

rep_means_A = np.asarray(rep_means_A, dtype=float)
rep_means_B = np.asarray(rep_means_B, dtype=float)
paired_diff = rep_means_A - rep_means_B

mean_d, hw_d = t_ci(paired_diff)
vrf = float((np.var(rep_means_A, ddof=1) + np.var(rep_means_B, ddof=1)) /
            np.var(paired_diff, ddof=1))

print(f'Policy A mean wait: {np.mean(rep_means_A):.3f} s')
print(f'Policy B mean wait: {np.mean(rep_means_B):.3f} s')
print(f'Paired A-B difference: {mean_d:.3f} ± {hw_d:.3f} s')
print(f'Variance-reduction factor: {vrf:.3f}')


## Task 5 — V&V Mini (10 pts)

Two short verification & validation checks:

1. **Conservation invariant.** At every moment, $\text{FLEET\_SIZE} = (\text{drones in air}) + (\text{drones in depot})$. The model exposes `state.max_invariant_violation` after a run — under correct code it is exactly $0$. Verify and record this value.

2. **Little's Law on the depot.** In steady state $L = \lambda W$ must hold. For the depot subsystem:
   - $L_{\text{depot}}$ = mean number of drones in the depot $= \text{FLEET\_SIZE} - \overline{(\text{in air})}$, sampled via `air_count_series(state)`.
   - $\lambda$ = arrival rate at the depot = (post-warmup completions) / (post-warmup duration).
   - $W_{\text{depot}}$ = mean per-drone sojourn (wait + service) in the depot. The model logs each completion as `(t_landed, swap_wait, test_wait, sojourn)` in `state.completions`.

   Compute both sides and report the ratio $L_{\text{predicted}} / L_{\text{observed}}$. It should be very close to $1$ (within 5%).

The conservation invariant is exactly satisfied: the maximum violation is 0, so the simulated fleet count is internally consistent. Little's Law also supports the steady-state output: the ratio $L_{predicted}/L_{observed}$ is approximately 1.000, well within the 5% tolerance. Together these checks do not prove the model is perfect, but they rule out two important implementation and accounting errors.


In [ ]:
# ── Your code below ─────────────────────────────────────────────────────
# A long replication is easiest for Little's Law. Reuse the long Policy A run
# from Task 3 when the notebook is run top-to-bottom.

state_vv = long_state_A if 'long_state_A' in globals() else run_replication(
    N_SWAP_A, N_TEST, MASTER_SEED, rep_index=0, duration=7 * LONG_RUN_DUR
)

invariant_violation = float(state_vv.max_invariant_violation)

t_air, n_air = air_count_series(state_vv)
post_air = t_air >= warmup_chosen
L_observed = FLEET_SIZE - float(np.mean(n_air[post_air]))

completions = np.asarray(state_vv.completions, dtype=float)
post_comp = completions[completions[:, 0] >= warmup_chosen]
run_duration = state_vv.env.now
arrival_rate = len(post_comp) / (run_duration - warmup_chosen)
W_depot = float(np.mean(post_comp[:, 3]))
L_predicted = arrival_rate * W_depot
analytical_bound = float(L_predicted / L_observed)

print(f'Max invariant violation: {invariant_violation:.0f}')
print(f'L_observed={L_observed:.3f}, lambda={arrival_rate:.5f}/s, W={W_depot:.3f}s')
print(f"Little's Law ratio L_predicted/L_observed = {analytical_bound:.4f}")


## Submit

Run the cell below at the very end. It writes `submission.json` for the autograder.


In [ ]:
# ── Do not modify. Writes submission.json. ─────────────────────────────
submission = {
    'classification':            classification,
    'warmup_welch':              int(warmup_welch) if warmup_welch is not None else -1,
    'warmup_mser':               int(warmup_mser)  if warmup_mser  is not None else -1,
    'warmup_chosen':             int(warmup_chosen) if warmup_chosen is not None else -1,
    'task3_rep_estimate':        float(mean_rep) if mean_rep is not None else float('nan'),
    'task3_rep_halfwidth':       float(hw_rep)   if hw_rep   is not None else float('nan'),
    'task3_R':                   int(R) if R is not None else -1,
    'task3_bm_estimate':         float(mean_bm) if mean_bm is not None else float('nan'),
    'task3_bm_halfwidth':        float(hw_bm)   if hw_bm   is not None else float('nan'),
    'task3_B':                   int(B_chosen) if B_chosen is not None else -1,
    'task3_K':                   int(K_chosen) if K_chosen is not None else -1,
    'task3_lag1':                float(lag1_chosen) if lag1_chosen is not None else float('nan'),
    'task4_paired_diff':         float(mean_d) if mean_d is not None else float('nan'),
    'task4_paired_halfwidth':    float(hw_d)   if hw_d   is not None else float('nan'),
    'task4_vrf':                 float(vrf)    if vrf    is not None else float('nan'),
    'task5_invariant_max_violation': float(invariant_violation) if invariant_violation is not None else float('nan'),
    'task5_analytical_bound':    float(analytical_bound) if analytical_bound is not None else float('nan'),
}
with open('submission.json', 'w') as f:
    json.dump(submission, f, indent=2)
print('submission.json written.')
